# Dataset protocol audit

This notebook validates explicit train/validation/test metadata and a bounded set of referenced files. It reads JSON registries but does not traverse dataset directories, decode images, modify data, or start training.

In [ ]:
import json
import os
import random
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import yaml


def required_directory(name):
    value = os.environ.get(name)
    assert value, f"Missing required environment variable: {name}"
    path = Path(value).expanduser().resolve()
    assert path.is_dir(), f"{name} is not a directory"
    return path


repo_root = required_directory("DFDHR_REPO_ROOT")
data_root = required_directory("DFDHR_DATA_ROOT")
runtime_root = required_directory("DFDHR_RUNTIME_ROOT")
json_root = Path(os.environ.get(
    "DFDHR_JSON_ROOT", repo_root / "preprocessing/dataset_json"
)).expanduser().resolve()
assert json_root.is_dir()
assert runtime_root != repo_root and repo_root not in runtime_root.parents
assert runtime_root != data_root and data_root not in runtime_root.parents
sys.path.insert(0, str(repo_root / "training"))
print("Git commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], text=True
).strip())


In [ ]:
from evaluation_utils import atomic_write_json, sha256_file

primary_name = "FaceForensics++"
primary_json_path = json_root / f"{primary_name}.json"
with primary_json_path.open(encoding="utf-8") as handle:
    primary_json = json.load(handle)[primary_name]

split_summary = {}
sampled_path_count = 0
for label_name, label_splits in primary_json.items():
    assert set(("train", "val", "test")).issubset(label_splits)
    video_ids_by_split = {}
    split_summary[label_name] = {}
    for split_name in ("train", "val", "test"):
        assert "c23" in label_splits[split_name]
        videos = label_splits[split_name]["c23"]
        assert videos, f"Empty {label_name}/{split_name}/c23 split"
        video_ids_by_split[split_name] = set(videos)
        frame_count = sum(len(info["frames"]) for info in videos.values())
        split_summary[label_name][split_name] = {
            "videos": len(videos),
            "frames": frame_count,
        }
        ordered_videos = sorted(videos.items())
        for _, info in (ordered_videos[0], ordered_videos[-1]):
            assert info["frames"]
            assert Path(info["frames"][0]).is_file()
            sampled_path_count += 1
    assert video_ids_by_split["train"].isdisjoint(video_ids_by_split["val"])
    assert video_ids_by_split["train"].isdisjoint(video_ids_by_split["test"])
    assert video_ids_by_split["val"].isdisjoint(video_ids_by_split["test"])

print("Primary JSON SHA-256:", sha256_file(primary_json_path))
print("Labels audited:", len(split_summary))
print("Bounded referenced files checked:", sampled_path_count)
print("Split summary:", split_summary)


In [ ]:
from dataset.abstract_dataset import DeepfakeAbstractBaseDataset

with (repo_root / "training/config/detector/dfd_hr.yaml").open(encoding="utf-8") as handle:
    config = yaml.safe_load(handle)
with (repo_root / "training/config/test_config.yaml").open(encoding="utf-8") as handle:
    config.update(yaml.safe_load(handle))
config["dataset_json_folder"] = str(json_root)
config["use_data_augmentation"] = False

datasets_by_mode = {}
for mode in ("train", "val", "test"):
    mode_config = config.copy()
    mode_config["train_dataset"] = [primary_name]
    mode_config["validation_dataset"] = primary_name
    mode_config["test_dataset"] = primary_name
    random.seed(config["manualSeed"])
    datasets_by_mode[mode] = DeepfakeAbstractBaseDataset(mode_config, mode=mode)

val_paths = set(datasets_by_mode["val"].image_list)
test_paths = set(datasets_by_mode["test"].image_list)
assert val_paths
assert test_paths
assert val_paths.isdisjoint(test_paths), "Validation resolved to test data"
loader_counts = {mode: len(dataset) for mode, dataset in datasets_by_mode.items()}
print("Dataset loader counts:", loader_counts)
print("Validation/test selected paths disjoint: True")


In [ ]:
external_name = os.environ.get("DFDHR_AUDIT_EXTERNAL_DATASET", "Celeb-DF-v2")
external_json_path = json_root / f"{external_name}.json"
assert external_json_path.is_file()
external_config = config.copy()
external_config["test_dataset"] = external_name
random.seed(config["manualSeed"])
external_dataset = DeepfakeAbstractBaseDataset(external_config, mode="test")
external_samples = sorted(set(external_dataset.image_list))[:8]
assert len(external_samples) == 8
assert all(Path(path).is_file() for path in external_samples)
print("External dataset role:", external_name)
print("External selected sample count:", len(external_dataset))
print("External bounded files checked:", len(external_samples))
print("External JSON SHA-256:", sha256_file(external_json_path))


In [ ]:
run_id = "dataset-protocol-audit_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
report_path = runtime_root / "jupyter-validation" / run_id / "audit.json"
atomic_write_json({
    "schema_version": 1,
    "git_commit": subprocess.check_output(
        ["git", "rev-parse", "HEAD"], text=True
    ).strip(),
    "primary_dataset_role": primary_name,
    "primary_json_sha256": sha256_file(primary_json_path),
    "split_summary": split_summary,
    "loader_counts": loader_counts,
    "validation_test_disjoint": True,
    "bounded_referenced_files_checked": sampled_path_count,
    "external_dataset_role": external_name,
    "external_json_sha256": sha256_file(external_json_path),
    "external_selected_sample_count": len(external_dataset),
    "external_bounded_files_checked": len(external_samples),
}, report_path)
print("Audit report role: DFDHR_RUNTIME_ROOT/jupyter-validation/<run-id>/audit.json")
